# Pandas Core with scikit-learn Breast Cancer Dataset

This notebook teaches **pandas core** using a **realistic ML dataset**: the `breast_cancer` dataset from `scikit-learn`.

We'll go step by step:
1. Load the dataset and convert it to a pandas DataFrame
2. Inspect structure, dtypes, and target
3. Select, filter, and transform columns
4. Handle missing values (even if this dataset is clean, we will simulate some)
5. Create features (vectorized transformations)
6. Grouping/aggregation (per target class)
7. Reshaping (pivot/melt) where it makes sense
8. Merging (synthetic example)
9. Time-like operations (on synthetic timestamps)
10. Interop with scikit-learn (separating features/target)

Everything is in **English** and focused on what you should master as an AI/ML practitioner.

## 1. Load the Breast Cancer Dataset
We will use `sklearn.datasets.load_breast_cancer`, which returns a bunch-like object with:
- `data`: numeric features
- `target`: 0/1 labels (malignant/benign)
- `feature_names`: column names
- `target_names`: names of the classes


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.utils import Bunch
import pandas as pd
import numpy as np

# load dataset as a Bunch so Pylance is happy
data_bunch: Bunch = load_breast_cancer()  # type: ignore

X = pd.DataFrame(data_bunch.data, columns=data_bunch.feature_names)
y = pd.Series(data_bunch.target, name="target")

df = pd.concat([X, y], axis=1)
df.head()

### 1.1 Dataset description
- This is a **binary classification** dataset.
- `target = 0` typically means **malignant**, `target = 1` means **benign** (we will confirm below).
- All features are numeric, which makes it perfect to practice pandas without worrying about text parsing.
- Even if the dataset is clean, we'll practice missing values and transformations because in real life your data won't be this clean.

In [ ]:
data_bunch.target_names

## 2. Inspect the DataFrame
First thing you should ALWAYS do in a notebook: check shape, dtypes, head, and basic stats.

In [ ]:
df.shape, df.dtypes.head()

In [ ]:
df.info()

In [ ]:
df.describe().T  # transpose for easier reading

### 2.1 Target balance
Check how many benign vs malignant samples there are.

In [ ]:
df["target"].value_counts()
df["target"].value_counts(normalize=True)

## 3. Selecting and Filtering
Even though all columns here are numeric, you still need to master column selection and row filtering.

In [ ]:
# Select a single column (Series)
df["mean radius"].head()

In [ ]:
# Select multiple columns (DataFrame)
df[["mean radius", "mean texture", "target"]].head()

### 3.1 Boolean filtering
Let's inspect only malignant cases (target == 0):

In [ ]:
malignant = df[df["target"] == 0]
benign = df[df["target"] == 1]
malignant.head()

You can combine conditions. For example, malignant cases with a large radius.

In [ ]:
df[(df["target"] == 0) & (df["mean radius"] > 20)][
    ["mean radius", "mean texture", "target"]
].head()

### 3.2 `.loc` and `.iloc`
You'll need these for more explicit selections.

In [ ]:
# label-based
df.loc[0, ["mean radius", "mean texture", "target"]]

# position-based
df.iloc[0:5, 0:5]

## 4. Working with dtypes
This dataset is already numeric, but we'll simulate a couple of scenarios you must know:
1. Converting numeric-looking strings to numbers
2. Creating a categorical column from the target


In [ ]:
# 4.1 Create a categorical version of the target
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})
df[["target", "target_name"]].head()

In [ ]:
# 4.2 Convert to category dtype (saves memory, faster groupby)
df["target_name"] = df["target_name"].astype("category")
df.dtypes.tail()

## 5. Handling Missing Data (with a simulated example)
The breast cancer dataset has no missing values, but in real-world AI pipelines you *must* handle them.
Let's create some missing values and show imputation patterns.

In [ ]:
df_na = df.copy()
df_na.loc[0:10, "mean radius"] = np.nan
df_na.loc[5:8, "mean texture"] = np.nan
df_na.isna().sum().head(10)

In [ ]:
# Impute numeric with median
df_imputed = df_na.copy()
num_cols = df_imputed.select_dtypes(include=["number"]).columns
for col in num_cols:
    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())
df_imputed.isna().sum().head(10)

## 6. Creating and Transforming Columns (Feature Engineering style)
We can create ratios, interaction features, logs, or flags.
This is one of the most important pandas skills for ML.

In [ ]:
df_feat = df.copy()

# Example feature: ratio of mean area to mean perimeter
df_feat["area_per_perimeter"] = df_feat["mean area"] / df_feat["mean perimeter"]

# Example feature: is large radius?
radius_threshold = df_feat["mean radius"].median()
df_feat["is_large_radius"] = (df_feat["mean radius"] > radius_threshold).astype(int)

df_feat[["mean radius", "area_per_perimeter", "is_large_radius"]].head()

You can also use `.assign()` to keep a clean pipeline-like style.

In [ ]:
df_feat2 = df.assign(
    radius_to_texture=lambda d: d["mean radius"] / d["mean texture"],
    is_benign=lambda d: (d["target"] == 1).astype(int),
)
df_feat2.head()

## 7. Grouping and Aggregations (`groupby`)
A very common pattern: *"show me stats per class"*. Here we group by `target_name`.

In [ ]:
agg_target = (
    df.groupby("target_name")
    .agg(
        n=("mean radius", "count"),
        avg_radius=("mean radius", "mean"),
        avg_texture=("mean texture", "mean"),
        avg_area=("mean area", "mean"),
    )
    .reset_index()
)
agg_target

In [ ]:
# Group by target and bin a numeric feature
df_bins = df.copy()
df_bins["radius_bin"] = pd.qcut(df_bins["mean radius"], q=4, labels=False)
df_bins.groupby(["target_name", "radius_bin"]).size().reset_index(name="count")

## 8. Reshaping (pivot / melt)
For this dataset, reshaping is less common, but it's still useful to know.
We'll create a small summary and pivot it.

In [ ]:
summary = (
    df.groupby("target_name")
    .agg(
        mean_radius=("mean radius", "mean"),
        mean_perimeter=("mean perimeter", "mean"),
        mean_smoothness=("mean smoothness", "mean"),
    )
    .reset_index()
)
summary

In [ ]:
long_summary = summary.melt(
    id_vars="target_name", var_name="metric", value_name="value"
)
long_summary.head()

In [ ]:
pivot_summary = long_summary.pivot_table(
    index="metric",
    columns="target_name",
    values="value",
)
pivot_summary

## 9. Merging / Joining (synthetic)
This dataset is single-table, so let's simulate a second table with model predictions or patient metadata and merge it.

In [ ]:
# Simulate an external table with patient_id and hospital
meta = pd.DataFrame(
    {
        "row_id": np.arange(len(df)),
        "hospital": np.random.choice(["H1", "H2", "H3"], size=len(df)),
    }
)

# Our main df currently has a RangeIndex (0..n-1), so we can merge on that
df_with_id = df.reset_index().rename(columns={"index": "row_id"})

merged = pd.merge(df_with_id, meta, on="row_id", how="left")
merged[["row_id", "hospital", "target_name"]].head()

## 10. Time-like Operations (synthetic)
Even though the breast cancer dataset has no timestamps, many ML problems do.
We'll create a fake timestamp column and show resampling and rolling.

In [ ]:
df_time = df.copy()
df_time["timestamp"] = pd.date_range("2025-01-01", periods=len(df_time), freq="H")
df_time = df_time.set_index("timestamp").sort_index()
df_time.head()

In [ ]:
# Resample to daily level and compute avg radius per day
daily = df_time.resample("D")["mean radius"].mean()
daily.head()

In [ ]:
# Rolling example
df_time["rolling_radius_24h"] = df_time["mean radius"].rolling("24H").mean()
df_time[["mean radius", "rolling_radius_24h"]].head(30)

## 11. `apply` vs Vectorization
You should avoid row-wise `apply` on large dataframes unless necessary. Vectorized/numpy ops are faster.

In [ ]:
def risk_score(row):
    # totally made-up rule for demo
    if (row["mean radius"] > 15) and (row["mean texture"] > 20):
        return "high"
    return "low"


df_apply = df.copy()
df_apply["risk"] = df_apply.apply(risk_score, axis=1)
df_apply[["mean radius", "mean texture", "risk"]].head()

In [ ]:
# Vectorized alternative (when possible):
df_vec = df.copy()
df_vec["risk"] = np.where(
    (df_vec["mean radius"] > 15) & (df_vec["mean texture"] > 20),
    "high",
    "low",
)
df_vec[["mean radius", "mean texture", "risk"]].head()

## 12. Interop with scikit-learn
Since this dataset **comes from scikit-learn**, it's natural to go back to it.
But in real projects, you clean & engineer in pandas → you extract X/y → you train with sklearn.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Separate features/target
X = df.drop(columns=["target", "target_name"], errors="ignore")
y = df["target"]

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train a simple model
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

print("Train score:", clf.score(X_train, y_train))
print("Test score:", clf.score(X_test, y_test))

## 13. Method-Chaining Style (clean ETL-like flow)
Let's build a single, readable transformation pipeline in pandas that:
1. Starts from the raw breast cancer df
2. Creates a couple of features
3. Aggregates by target
4. Returns a compact table

In [ ]:
summary_by_target = (
    df.assign(
        radius_ratio=lambda d: d["mean radius"] / d["worst radius"],
        is_benign=lambda d: (d["target"] == 1).astype(int),
    )
    .groupby("target_name")
    .agg(
        n=("mean radius", "count"),
        avg_radius=("mean radius", "mean"),
        avg_ratio=("radius_ratio", "mean"),
        benign_rate=("is_benign", "mean"),
    )
    .reset_index()
)
summary_by_target

## 14. What you should master from this notebook
- Loading a dataset and converting to pandas
- Inspecting: `.head()`, `.info()`, `.describe()`
- Selecting columns/rows: `[]`, `.loc`, `.iloc`
- Boolean filtering with multiple conditions
- Creating new columns (vectorized and with `.assign()`)
- Handling missing values (`.isna()`, `.fillna()`, `.dropna()`)
- Grouping and aggregating with named aggregations
- Reshaping small summaries with `melt` and `pivot_table`
- Merging two DataFrames (`pd.merge`)
- Creating synthetic datetime columns and resampling
- Avoiding slow `.apply(axis=1)` when possible
- Extracting `X` and `y` for scikit-learn
- Writing transformations in a method-chaining style